# BERT Fine-Tuning — Symptom2Disease (Colab GPU)

This notebook fine-tunes `bert-base-uncased` on the Symptom2Disease dataset
to act as **AI Technique 3** in the coursework complementing the
TF-IDF + LinearSVC and Word+Char + Logistic Regression models trained in
`src/train_models.py`.

**How to use**
1. Upload `dataset/Symptom2Disease.csv` to your Colab session (Files).
2. Set Runtime → Change runtime type → GPU.
3. Run all cells.
4. Download the generated `bert_meta.json` and place it in `artifacts/`.

After that, re-running `python -m src.train_models` will pick up the BERT
metrics automatically and include them in `model_comparison.json`.

In [ ]:
# Install dependencies (Colab usually has torch already)
!pip install -q transformers==4.44.2 datasets==2.21.0 evaluate==0.4.2 scikit-learn==1.5.1 accelerate==0.34.2

In [ ]:
import os, json, random, numpy as np, pandas as pd, torch
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding,
                          set_seed)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)

print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Load dataset (Colab path — adjust if you've put it elsewhere)
CSV = 'Symptom2Disease.csv'
if not os.path.exists(CSV):
    CSV = '../dataset/Symptom2Disease.csv' # Repo path

print(f'Loading dataset from: {CSV}')
df = pd.read_csv(CSV)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
df = df[['label', 'text']].dropna().reset_index(drop=True)
df['label_norm'] = df['label'].str.lower().str.strip()

label2id = {l: i for i, l in enumerate(sorted(df['label_norm'].unique()))}
id2label = {i: l for l, i in label2id.items()}
df['label_id'] = df['label_norm'].map(label2id)

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['text'].tolist(),
    df['label_id'].tolist(),
    test_size=0.30, random_state=SEED, stratify=df['label_id'],
)
print(f'Train {len(X_train_txt)}  Test {len(X_test_txt)}  Classes {len(label2id)}')


In [ ]:
MODEL_NAME = 'bert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    return tok(batch['text'], truncation=True, max_length=128)

train_ds = Dataset.from_dict({'text': X_train_txt, 'label': y_train}).map(encode, batched=True)
test_ds  = Dataset.from_dict({'text': X_test_txt,  'label': y_test }).map(encode, batched=True)
collator = DataCollatorWithPadding(tokenizer=tok)


In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'precision_macro': pr,
        'recall_macro': rc,
        'f1_macro': f1,
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir='./bert_out',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=20,
    seed=SEED,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
metrics = trainer.evaluate()
print('Final metrics:', metrics)

bert_meta = {
    'model': MODEL_NAME,
    'accuracy': float(metrics['eval_accuracy']),
    'precision_macro': float(metrics['eval_precision_macro']),
    'recall_macro': float(metrics['eval_recall_macro']),
    'f1_macro': float(metrics['eval_f1_macro']),
    'epochs': int(args.num_train_epochs),
    'batch_size': int(args.per_device_train_batch_size),
    'learning_rate': float(args.learning_rate),
    'trained_at': datetime.utcnow().isoformat() + 'Z',
}
with open('bert_meta.json', 'w') as f:
    json.dump(bert_meta, f, indent=2)
print('\nWrote bert_meta.json — download this file and place it in artifacts/.')


In [ ]:
# Save the fine-tuned model so you can reuse it locally.
# trainer.save_model('bert_symptom2disease')
# tok.save_pretrained('bert_symptom2disease')
# !zip -r bert_symptom2disease.zip bert_symptom2disease
